In [ ]:
import cv2
import pyautogui
import numpy as np
import playsound
import time

button_img = cv2.imread('skipad.png', cv2.IMREAD_GRAYSCALE)
w, h = button_img.shape[::-1]

cap = cv2.VideoCapture(1, cv2.CAP_DSHOW)

if not cap.isOpened():
    print("Error: Could not access the webcam.")
    exit()

while True:
    ret, frame = cap.read()

    if not ret:
        print("Failed to capture frame.")
        break

    gray_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    # increase contrast
    gray_frame = cv2.convertScaleAbs(gray_frame, alpha=1.5, beta=0)
    #crop the frame to bottom right corner and draw rectangle to show where we are looking for the button
    h_frame, w_frame = gray_frame.shape
    crop_x_start = w_frame - 300
    crop_y_start = h_frame - 80
    cropped_frame = gray_frame[crop_y_start:h_frame, crop_x_start:w_frame]
    cv2.rectangle(frame, (crop_x_start, crop_y_start), (w_frame, h_frame), (255, 0, 0), 2)
    
    result = cv2.matchTemplate(cropped_frame, button_img, cv2.TM_CCOEFF_NORMED)
    threshold = 0.9  # higher threshold to reduce false pos

    locations = np.where(result >= threshold)

    if len(locations[0]) > 0:
        for pt in zip(*locations[::-1]):
            cv2.rectangle(frame, pt, (pt[0] + w, pt[1] + h), (0, 255, 0), 2)
        
        print("Ad detected, skip it!")
        playsound.playsound('retrobeeps.wav')  

        # if it's on pc, autoclick 
        # pyautogui.click(pt[0] + w // 2, pt[1] + h // 2)
        # time.sleep(2)  # delay to prevent rapid clicking

    cv2.imshow('Webcam Feed', frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
